# 03 — K-Nearest Neighbors From First Principles

This notebook implements KNN from scratch for classification and regression.

In [ ]:
import numpy as np

## 1. Helper Functions

In [ ]:
def train_test_split_numpy(X, y, test_size=0.25, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    indices = rng.permutation(n)
    test_n = int(n * test_size)
    test_idx = indices[:test_n]
    train_idx = indices[test_n:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std = np.where(std == 0, 1, std)
    return (X_train - mean) / std, (X_test - mean) / std

## 2. Euclidean Distance

$$
d(x,y)=\sqrt{\sum_j(x_j-y_j)^2}
$$

In [ ]:
def euclidean_distances(X_train, X_query):
    diff = X_query[:, None, :] - X_train[None, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=2))

## 3. KNN Classification

In [ ]:
def knn_predict_classification(X_train, y_train, X_query, k=5):
    distances = euclidean_distances(X_train, X_query)
    neighbor_indices = np.argsort(distances, axis=1)[:, :k]
    neighbor_labels = y_train[neighbor_indices]
    predictions = []
    for labels in neighbor_labels:
        counts = np.bincount(labels.astype(int))
        predictions.append(np.argmax(counts))
    return np.array(predictions)


def accuracy_score(y_true, y_pred):
    return np.mean(y_true == y_pred)

## 4. Classification Example

In [ ]:
rng = np.random.default_rng(42)

n = 240
class0 = rng.multivariate_normal([-1.5, -1.0], [[0.8, 0.2], [0.2, 0.7]], size=n // 2)
class1 = rng.multivariate_normal([1.4, 1.1], [[0.9, -0.25], [-0.25, 0.9]], size=n // 2)

X = np.vstack([class0, class1])
y = np.array([0] * (n // 2) + [1] * (n // 2))

X_train, X_test, y_train, y_test = train_test_split_numpy(X, y)
X_train_scaled, X_test_scaled = standardize_train_test(X_train, X_test)

for k in [1, 3, 5, 11, 21]:
    pred = knn_predict_classification(X_train_scaled, y_train, X_test_scaled, k=k)
    print(k, accuracy_score(y_test, pred))

## 5. KNN Regression

$$
\hat{y}=\frac{1}{k}\sum_{j=1}^{k}y_{(j)}
$$

In [ ]:
def knn_predict_regression(X_train, y_train, X_query, k=5):
    distances = euclidean_distances(X_train, X_query)
    neighbor_indices = np.argsort(distances, axis=1)[:, :k]
    neighbor_values = y_train[neighbor_indices]
    return neighbor_values.mean(axis=1)


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

x = np.linspace(0, 10, 120)
y_reg = np.sin(x) + 0.18 * x + rng.normal(0, 0.25, size=len(x))
X_reg = x.reshape(-1, 1)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split_numpy(X_reg, y_reg)

for k in [1, 3, 9, 21]:
    pred_r = knn_predict_regression(X_train_r, y_train_r, X_test_r, k=k)
    print(k, mae(y_test_r, pred_r))

## 6. Weighted KNN Regression

In [ ]:
def knn_predict_weighted_regression(X_train, y_train, X_query, k=5, eps=1e-8):
    distances = euclidean_distances(X_train, X_query)
    neighbor_indices = np.argsort(distances, axis=1)[:, :k]
    predictions = []
    for row_id, idx in enumerate(neighbor_indices):
        neighbor_distances = distances[row_id, idx]
        neighbor_values = y_train[idx]
        weights = 1 / (neighbor_distances + eps)
        predictions.append(np.sum(weights * neighbor_values) / np.sum(weights))
    return np.array(predictions)

pred_weighted = knn_predict_weighted_regression(X_train_r, y_train_r, X_test_r, k=9)
mae(y_test_r, pred_weighted)

## Reflection

KNN does not learn weights. It uses the geometry of the dataset itself as the model.